<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# DATA CLEANING: PREPARING THE AMES HOUSING DATASET

<br>

**About:** A hands-on walkthrough of cleaning and merging the Ames Housing dataset with macroeconomic indicators, covering inspection, temporal aggregation, and validation.

**Learning Goals:** After completing this notebook, you will be able to:

- Load and inspect the Ames Housing dataset to identify data quality issues
- Handle missing values using principled strategies (not just deletion)
- Aggregate monthly macroeconomic data to quarterly periods for alignment
- Merge macroeconomic indicators with housing sales by time period
- Validate a cleaned dataset before passing it downstream

**Keywords:** data cleaning, temporal aggregation, data merging, missing values, data validation

**Prerequisite Knowledge:** (1) Basic Python and pandas, (2) Familiarity with DataFrames

**Target User:** Learners with Python experience who want to understand real-world data cleaning workflows

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: UNDERSTANDING THE AMES DATASET](#Part_1)
> #### [PART 2: LOADING AND INSPECTING DATA](#Part_2)
> #### [PART 3: TEMPORAL AGGREGATION](#Part_3)
> #### [PART 4: MERGING MACRO INDICATORS](#Part_4)
> #### [PART 5: VALIDATION AND EXPORT](#Part_5)

<br>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries loaded.")

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **UNDERSTANDING** the Ames Dataset

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

The Ames Housing dataset contains sales records for residential properties in Ames, Iowa. Each row is one home sale, with columns covering physical features (square footage, bedrooms, garage), condition ratings, and the sale price.

What makes this project unusual is that we augment Ames with **macroeconomic indicators** - quarterly measurements of regional economic health: unemployment rate, manufacturing employment, income after taxes, and sales tax revenue. The core question: do broader economic conditions improve price prediction beyond what property features alone can explain?

**Why macro indicators?** Most housing models use only "micro" features - what you are buying. The hypothesis here is that *when* you are buying matters too. A home sold during a recession may clear at a lower price than the identical home sold during a boom, holding all physical features constant.

___

**Note:** The Ames dataset is well-documented in De Cock (2011), "Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project," *Journal of Statistics Education*. Macro indicators were sourced from public FRED (Federal Reserve Economic Data) series for Iowa. Sources: [AmesHousing R package](https://cran.r-project.org/package=AmesHousing), [FRED](https://fred.stlouisfed.org/).

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Why might economic indicators improve price prediction beyond property features alone? Describe one mechanism by which regional unemployment could affect the sale price of a home with fixed physical characteristics.**

<br>

```python
# Write your explanation in comments below:
# mechanism = "..."
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **LOADING** and **INSPECTING** Data

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Why Inspection Comes First

Before cleaning, you need to *see* what you are working with. Inspection tells you:
- How many rows and columns you have
- Which columns have missing values and how many
- Data types (numeric, text, date) - a price stored as a string will silently fail numeric operations
- Basic statistics (mean, min, max) that reveal outliers or encoding errors

This order prevents wasted effort: you might spend time imputing a column that turns out to be irrelevant, or miss that a column encoding a category as integers will mislead a model.

<br>

**Inspection Workflow**

In [ ]:
# Create a synthetic Ames-like dataset for demonstration
np.random.seed(42)
n = 1460  # Ames dataset has 1460 rows

ames = pd.DataFrame({
    "SalePrice": np.random.lognormal(mean=12.0, sigma=0.4, size=n).round(-2),
    "TotalSqFt": np.random.normal(1500, 400, n).clip(400, 4000).round(),
    "YearBuilt": np.random.randint(1900, 2010, n),
    "GarageCars": np.random.choice([0, 1, 2, 3], n, p=[0.05, 0.2, 0.6, 0.15]),
    "OverallCond": np.random.randint(1, 10, n),
    "Year": np.random.randint(2006, 2011, n),
    "Month": np.random.randint(1, 13, n),
})

# Introduce realistic missing values (~3-5% in select columns)
ames.loc[np.random.choice(n, 70, replace=False), "GarageCars"] = np.nan
ames.loc[np.random.choice(n, 15, replace=False), "TotalSqFt"] = np.nan

# Step 1: Shape
print(f"Shape: {ames.shape}")  # (rows, columns)

# Step 2: Data types
print("\nData types:")
print(ames.dtypes)

# Step 3: Missing values
missing = ames.isnull().sum()
print("\nMissing values:")
print(missing[missing > 0])

# Step 4: Summary statistics
print("\nSummary statistics:")
print(ames.describe().round(1))

### Why This Order?

We inspect in this order because each step informs the next:
- **Shape first**: Gives you the overall picture - 1460 rows, 7 columns.
- **Types second**: A column stored as `object` when you expect `float64` signals an encoding problem. Fix this before aggregating.
- **Missing third**: You can't decide how to handle nulls without knowing data types. A 3.4% null rate in `GarageCars` might mean "no garage," not missing data at all.
- **Statistics last**: Reveals outliers. A `TotalSqFt` minimum of 50 suggests a data entry error; a `SalePrice` maximum of $10M in Ames is impossible and would skew modeling.

<strong style="color:red">KEY CONSIDERATION:</strong> Never impute or drop missing values before understanding *why* they are missing. Random missingness (MCAR) justifies mean imputation; systematic missingness (MAR/MNAR) requires a different strategy - for example, a null `GarageCars` might always mean the property has no garage, and should be filled with 0.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `GarageCars` column has 70 missing values out of 1460 (~4.8%). Write code to check whether those missing values all come from homes built before 1950. If they do, what does that tell you about how to handle the nulls?**

<br>

```python
# Investigate the pattern of missing GarageCars values:
missing_garage_mask = ames["GarageCars"].isnull()
# YOUR CODE HERE
# missing_year_built = ...
# print(missing_year_built.describe())
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **TEMPORAL** Aggregation

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### The Problem: Frequency Mismatch

Macroeconomic data often arrives at a different frequency than the event data you want to merge it with. Here:
- **Housing sales**: recorded by quarter (four data points per year)
- **Macro indicators**: recorded monthly by FRED (twelve data points per year)

To merge them, you must first bring macro data to the same frequency - quarterly. The choice of aggregation method matters.

**Why average, not sum?** Unemployment rate is a *rate*, not a count. The quarterly unemployment rate should reflect the average joblessness across those three months. Summing rates would give a meaningless number (a rate of 15% for a quarter when each month was 5%). Summing is appropriate for *counts* like total new home permits issued.

___

**Note:** This frequency-mismatch problem is standard in macroeconomic data work. See: Ghysels, E. & Marcellino, M. (2018), *Applied Economic Forecasting Using Time Series Methods*, Chapter 3. Source: [Oxford University Press](https://global.oup.com/academic/product/applied-economic-forecasting-using-time-series-methods-9780190622015).

___

In [ ]:
def aggregate_monthly_to_quarterly(monthly_values):
    """
    Convert a list of monthly values to quarterly by averaging.

    Each group of 3 consecutive months becomes one quarter:
    Q1 = mean(Jan, Feb, Mar), Q2 = mean(Apr, May, Jun), etc.

    Args:
        monthly_values: list or array with length divisible by 3

    Returns:
        List of quarterly averages (length = len(monthly_values) // 3)
    """
    quarterly = []
    for i in range(0, len(monthly_values), 3):
        quarter_avg = np.mean(monthly_values[i:i+3])
        quarterly.append(round(quarter_avg, 2))
    return quarterly

# Simulate 24 months (2 years) of monthly unemployment data for Iowa
monthly_unemployment = [4.8, 4.7, 4.6, 4.5, 4.4, 4.3,
                        4.2, 4.1, 4.2, 4.3, 4.5, 4.7,
                        5.0, 5.2, 5.3, 5.1, 4.9, 4.8,
                        4.6, 4.5, 4.4, 4.3, 4.2, 4.1]

quarterly_unemployment = aggregate_monthly_to_quarterly(monthly_unemployment)

print("Monthly unemployment (first 12):", monthly_unemployment[:12])
print("\nQuarterly unemployment (first year):")
for i, q in enumerate(quarterly_unemployment[:4]):
    months = monthly_unemployment[i*3:(i+1)*3]
    print(f"  Q{i+1}: {months} -> average = {q}%")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `aggregate_monthly_to_quarterly` function uses `np.mean`. A colleague suggests using the last month of each quarter instead (e.g., March for Q1). Modify the function to use the last-month strategy, then compare the two outputs on the unemployment data above. When would last-month be a better choice than averaging?**

<br>

```python
def aggregate_monthly_to_quarterly_last(monthly_values):
    ### YOUR CODE HERE ###
    quarterly = ...
    return quarterly

# Compare outputs:
# avg_result = aggregate_monthly_to_quarterly(monthly_unemployment)
# last_result = aggregate_monthly_to_quarterly_last(monthly_unemployment)
# print("Average:", avg_result[:4])
# print("Last month:", last_result[:4])
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **MERGING** Macro Indicators

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Combining Datasets by Time Period

Once both datasets are quarterly, we merge them by matching year and quarter. Each housing sale is labeled with the quarter it occurred in, then joined to the macro indicators for that quarter.

This is a **left merge**: every housing sale row is preserved, and macro values are attached wherever a matching quarter exists. Rows without a match get `NaN` macro values - a signal to investigate whether your quarterly coverage is complete.

In [ ]:
def get_quarter(month):
    """Return quarter number (1-4) for a given month (1-12)."""
    return (month - 1) // 3 + 1

# Assign quarter to each housing sale
ames["Quarter"] = ames["Month"].apply(get_quarter)

# Build a macro indicators table (2 years x 4 quarters = 8 rows)
years = [2006, 2006, 2006, 2006, 2007, 2007, 2007, 2007,
         2008, 2008, 2008, 2008, 2009, 2009, 2009, 2009,
         2010, 2010, 2010, 2010]
quarters = [1, 2, 3, 4] * 5
np.random.seed(0)
macro = pd.DataFrame({
    "Year": years,
    "Quarter": quarters,
    "UnemploymentRate": np.round(np.random.uniform(4.0, 6.5, 20), 1),
    "MfgEmployment": np.random.randint(85000, 100000, 20),
})

# Merge: each home sale gets the macro conditions of its quarter
ames_enriched = ames.merge(macro, on=["Year", "Quarter"], how="left")

print(f"Before merge: {ames.shape}")
print(f"After merge:  {ames_enriched.shape}")
print(f"\nNull macro values after merge:")
print(ames_enriched[["UnemploymentRate", "MfgEmployment"]].isnull().sum())
print("\nSample rows:")
print(ames_enriched[["SalePrice", "Year", "Quarter", "UnemploymentRate", "MfgEmployment"]].head(5))

### Why This Matters for Prediction

After merging, each row in `ames_enriched` contains:
- Property features: `TotalSqFt`, `YearBuilt`, `GarageCars`, `OverallCond`
- Economic context: `UnemploymentRate`, `MfgEmployment` for the quarter the home sold

A model trained on `ames_enriched` can now test whether regional economic conditions improve prediction beyond property features alone - the core research question of this project.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **A home was sold on April 15, 2009. After running the merge above, check what macro values it received. Then: if April 15 were instead a missing `Month` value (NaN), what would happen to the `Quarter` and macro columns for that row? Fix the code to handle this case.**

<br>

```python
# Find a sale from April 2009 in ames_enriched:
# sample = ames_enriched[(ames_enriched["Year"] == 2009) & (ames_enriched["Month"] == 4)]
# print(sample[["SalePrice", "Year", "Quarter", "UnemploymentRate"]].head(3))

# YOUR CODE HERE: handle NaN Month values before computing Quarter
# ames["Quarter"] = ...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_5'></a>

<hr style="border: 2px solid#003262;" />

#### PART 5

## **VALIDATION** and Export

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Final Validation Checklist

Before exporting cleaned data for modeling, run programmatic checks. A checklist you eyeball is slower and less reliable than assertions that fail loudly.

Checks to run:
1. **No unexpected nulls**: Know which columns are allowed to have nulls and which are not.
2. **Correct types**: Prices must be numeric; quarters must be integers 1-4.
3. **Reasonable ranges**: No negative prices. No `TotalSqFt` of 50 for a residential property.
4. **No duplicates**: A home sold twice in the same quarter is suspicious.
5. **Merge completeness**: The merged row count should match the original housing row count (left merge).

In [ ]:
def validate_cleaned_data(df):
    """
    Run validation checks on the merged, cleaned dataset.
    Raises AssertionError with a descriptive message if any check fails.
    Returns True if all checks pass.
    """
    # Check 1: No duplicate rows
    n_dupes = df.duplicated().sum()
    assert n_dupes == 0, f"Found {n_dupes} duplicate rows"

    # Check 2: SalePrice is positive
    assert (df["SalePrice"] > 0).all(), "Found non-positive sale prices"

    # Check 3: Year is in expected range for Ames data
    assert df["Year"].between(1900, 2030).all(), "Year out of expected range"

    # Check 4: Quarter is 1-4
    assert df["Quarter"].dropna().isin([1, 2, 3, 4]).all(), "Quarter values outside 1-4"

    # Check 5: TotalSqFt is plausible (residential: at least 200 sqft)
    non_null_sqft = df["TotalSqFt"].dropna()
    assert (non_null_sqft > 200).all(), "Found implausibly small TotalSqFt"

    print("All validation checks passed.")
    return True

validate_cleaned_data(ames_enriched)

# Export for next notebook:
# ames_enriched.to_csv("data/ames_with_macro_clean.csv", index=False)
# print("Exported to data/ames_with_macro_clean.csv")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Add a sixth validation check to `validate_cleaned_data`: confirm that `UnemploymentRate` values, where not null, fall within a plausible range (say, 0% to 30%). Then deliberately introduce a bad row and verify the check raises an error.**

<br>

```python
# Add the UnemploymentRate range check inside a copy of the function:

# Introduce a bad row to test it:
# bad_df = ames_enriched.copy()
# bad_df.loc[0, "UnemploymentRate"] = 150.0  # impossible rate
# validate_cleaned_data(bad_df)  # should raise AssertionError
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

---

## Summary

Data cleaning transforms raw data into a form ready for analysis:
- **Inspect first** to understand structure, types, and missing data
- **Aggregate** to a common time frequency before merging
- **Merge** by time period to combine complementary datasets
- **Validate** programmatically before handing off

In the next notebook (`02_exploratory_analysis.ipynb`), we visualize and understand the cleaned data before building models.

<hr style="border: 6px solid#003262;" />